In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
os.environ["PYKEOPS_VERBOSE"] = "0"
import sys
sys.path.append(os.path.abspath("conditional-flow-matching"))
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from visualize.plots import *

In [3]:
import random
import umap
from models.modules import *
from visualize.plots import *

In [4]:
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
from scripts.run_model import *
from eval.eval import *

In [6]:
%reload_ext autoreload
%autoreload 2

In [7]:
############################################################

In [8]:
energy_only = False

In [9]:
# #DEBUG: many values changed to assess the 5-PC + straight baseline
# config = Config(
#     {        
#         "model_class": "metricflow",
#         "score_max_epochs": 100,
#         "energy_max_epochs": 100,
#         "embed_max_epochs": 2 if energy_only else 5000,
#         "flow_max_epochs": 2 if energy_only else 5000,
        
#         "lr": 1e-4,
#         "dropout": 0.0,
#         "pc_dim": 50,
#         "cond_dim": 50,
#         # "pc_dim": 5,
#         # "cond_dim": 5,
#         "hidden_dim": 256,
#         "batch_size": 512,
#         "num_freq": 32,
#         "num_layers": 3,
#         "control_only": True,
#         "force_cpu": False,
#         "constrain": True,
#         "gradient_clip_val": 10.0,
#         "loader_batch_size": 20,
#         "accumulate_grad_batches": 1,
#         "warmup_steps": 0,
#         "ema_decay": .999,
#         "score_beta": 1.0,

#         "num_sigmas": 10,
#         "sigma_min": 0.1,
#         "sigma_max": 0.5,
        
#         "geo_sigma": 0.0,

#         "latent_dim": 100,

#         "skip": True,
#         "rescale": 0.5,
#         # "rescale": 0.0, #DEBUG FOR STRAIGHT GEODESICS
        
#         "pre_low_q": .05,
#         "pre_high_q": .95,
#         "low_q": .05,
#         "high_q": .99,

#         "weight_beta": 0.5,

#         "beta_low": 1.0,
#         "beta_high": 1.0,
#         "gamma": 0.2,
#         "margin": 1.0,

#         "sigma": 0.05,
#         "cfg_p_u": 0.0,
#         "cfg_w": 1.0,
#     }
# )


config = Config(
    {        
        "model_class": "metricflow",
        "score_max_epochs": 200,
        "energy_max_epochs": 200,
        "embed_max_epochs": 2 if energy_only else 500,
        "flow_max_epochs": 2 if energy_only else 500,
        "lr": 1e-4,
        "dropout": 0.0,
        "pc_dim": 50,
        "cond_dim": 50,
        "hidden_dim": 256,
        "batch_size": 256,
        "num_freq": 32,
        "num_layers": 4,
        "control_only": True,
        "force_cpu": False,
        "constrain": True,
        "gradient_clip_val": 0,
        "loader_batch_size": 6,
        "warmup_steps": 0,
        # "ema_decay": None,
        "ema_decay": 1-1e-3,

        "pita_steps": 1,

        "fast_ot": True,

        "num_sigmas": 20,
        "sigma_min": 0.05,
        "sigma_max": 0.2,

        "score_beta": 10.0,
        "num_eigs": 3,

        "score_alpha": 0.99,
        "knn_interpolation": False,
        
        "geo_sigma": 0.0,

        "latent_dim": 100,

        "skip": True,
        "rescale": 0.5,
        # "rescale": 0.0, #Straight flow

        "pre_low_q": .05,
        "pre_high_q": .95,
        "low_q": .05,
        "high_q": .90,

        "weight_beta": 0.3,

        "beta_low": 1.0,
        "beta_high": 1.0,
        "gamma": 0.2,
        "margin": 1.0,

        "sigma": 0.1,
        "cfg_p_u": 0.0,
        "cfg_w": 1.0,
    }
)

project = "fm-ipynb"

In [10]:
project = "zebrafish"

In [11]:
wandb.finish()

In [12]:
print(config.model_class)

metricflow


In [13]:
from collections import namedtuple
Holdout = namedtuple('Holdout', ['t', 'gene'])
holdout = Holdout(24, 'tbx16-tbx16l')
print(holdout.gene)

tbx16-tbx16l


In [14]:
adata, values = process_data(pc_dim=config.pc_dim, data="zebrafish") #TODO: make condition embeddings one-hot for simplicity

/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:379: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  adata.obsm[key_obsm] = X_pca


In [15]:
adata[adata.obs['gene_target'] == 'tbx16-tbx16l'].obs['timepoint'].unique()

array([24, 18, 36])

In [16]:
# values

In [17]:
if config.constrain:
    values = ['ctrl-inj', holdout.gene]
    adata = adata[adata.obs['gene_target'].isin(values)]
    adata = adata[adata.obs['timepoint'].isin([18, 24, 36])]

In [18]:
# adata.uns['std'] = np.std(adata.obsm['X_pca'])
# adata_raw = adata.copy()
# adata.obsm['X_pca'] /= adata.uns['std']

adata.uns['std'] = np.std(adata.obsm['X_pca'], axis=0, keepdims=True)
adata_raw = adata.copy()
adata.obsm['X_pca'] /= adata.uns['std']

# adata.uns['std'] = 1.0
# adata_raw = adata.copy()


/tmp/ipykernel_2217711/3560932423.py:1: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns['std'] = np.std(adata.obsm['X_pca'])


In [19]:
test_bool = (adata.obs['timepoint'] == holdout.t) & (adata.obs['gene_target'] == holdout.gene)
adata_train = adata[~test_bool]
adata_test = adata[test_bool]

In [20]:
conditions, dataset = extract_dataset(adata_train, values)

In [21]:
pre_score_models, pre_energy_models, score_model, energy_model, embed_model, flow_model = run_full_model(config, project, adata, values, conditions, dataset)

Running phase score:.......


wandb: Currently logged in as: az831 (az831-new-york-genome-center) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
You are using a CUDA device ('NVIDIA GeForce RTX 4080 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type | Params | Mode 
-------------------------------------------
0 | score_net | EMA  | 

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█████████████████
train_loss,█▆▆█▇▆▇▆▄▇▄▆▅▆▅▂▃▂▄▃▄▃▃█▃▅▅▂▇▁▃▁▃▂▆▃▃▆▅▂
trainer/global_step,▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇████
epoch,1
train_loss,1272.58093
trainer/global_step,2291


Running phase energy:.......


...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type           | Params | Mode 
------------------------------------------------------
0 | embed_net  | SimpleDenseNet | 238 K  | train
1 | geo_net    | SinNet         | 267 K  | train
2 | energy_net | SimpleScoreNet | 159 K  | eval 
3 | score_net  | EMA            | 318 K  | eval 
------------------------------------------------------
505 K     Trainable params
477 K     Non-trainable params
982 K     Total params
3.930     Total estimated model params 

epoch,▁█
train_loss,▁█
trainer/global_step,▁█
epoch,1
train_loss,428.38647
trainer/global_step,1


Running phase flow:.......


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type           | Params | Mode 
-----------------------------------------------------
0 | flow_net  | SinNet         | 254 K  | train
1 | embed_net | SimpleDenseNet | 238 K  | eval 
2 | geo_net   | SinNet         | 267 K  | eval 
3 | score_net | EMA            | 318 K  | eval 
-----------------------------------------------------
521 K     Trainable params
556 K     Non-trainable params
1.1 M     Total params
4.312     Total estimated model params size (M

epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇██
train_loss,█▆▇▄▅▅▅▅▅▂▄▄▄▃▄▃▃▅▃▃▄▃▆▁▁▄▂▄▅▅▃▂▁▄▂▄▃▃▄▃
trainer/global_step,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇██
epoch,499
train_loss,6.8431
trainer/global_step,499


In [22]:
from visualize.plots import *

In [23]:
def visualize_energy(
    embed_model,
    e_model,
    samples,                       # AnnData OR torch.Tensor [N,D]
    *,
    # --- AnnData-specific options ---
    layer=None,
    use_rep=None,
    obsm_key="X_umap",
    recompute_umap=False,
    copy=False,
    # --- energy options ---
    prune=False,
    prune_attr="G_interpolate",
    invert=False,
    color_clip=None,
    # --- plotting / compute options ---
    subsample=None,
    batch_size=512,
    device=None,
    random_state=0,
    cmap="inferno",
    kde_cmap="viridis",            # NEW: colour‑map for density panel
    show=True,
    **scanpy_plot_kwargs,
):
    import numpy as np
    import torch
    import matplotlib.pyplot as plt
    from scipy.stats import gaussian_kde         # ← NEW

    try:
        import scanpy as sc
    except ImportError as e:
        raise ImportError(
            "visualize_energy requires scanpy.  `pip install scanpy`"
        ) from e

    # ------------------------------------------------------------------ #
    # helper for mini‑batch evaluation (unchanged)                       #
    # ------------------------------------------------------------------ #
    def _eval_model_in_batches(tensor_x, fn, bs):
        outs = []
        with torch.no_grad():
            for s in range(0, tensor_x.shape[0], bs):
                outs.append(fn(tensor_x[s : min(s + bs, tensor_x.shape[0])]))
        return torch.cat(outs, dim=0)

    # ------------------------------------------------------------------ #
    # Path A – AnnData (tensor path omitted for brevity)                 #
    # ------------------------------------------------------------------ #
    if hasattr(samples, "X") and not torch.is_tensor(samples):
        adata = samples.copy() if copy else samples
        # ---------- choose input matrix (unchanged) ----------
        if layer is not None and use_rep is not None:
            raise ValueError("Specify at most one of `layer` or `use_rep`.")
        if use_rep is not None:
            X_in = adata.obsm[use_rep]
        elif layer is not None:
            X_in = adata.layers[layer]
        else:
            X_in = adata.X
        X_np = X_in.toarray() if hasattr(X_in, "toarray") else np.asarray(X_in)

        # ---------- energy / metric tensor ----------
        x_tensor = torch.as_tensor(X_np, dtype=torch.get_default_dtype())
        if device is None:
            device = next(embed_model.parameters()).device
        x_tensor = x_tensor.to(device)

        if prune:
            fn = getattr(embed_model, prune_attr)
        else:
            fn = e_model.get_energy
        scalar = _eval_model_in_batches(x_tensor, fn, batch_size).flatten()
        # scalar = np.exp(-scalar)
        scalar_np = scalar.detach().cpu().numpy()

        # ---------- colour clipping ----------
        vmin = vmax = None
        if color_clip is not None:
            if isinstance(color_clip, tuple) and len(color_clip) == 3 and color_clip[0] == "q":
                vmin = float(np.percentile(scalar_np, color_clip[1]))
                vmax = float(np.percentile(scalar_np, color_clip[2]))
            elif isinstance(color_clip, (tuple, list)) and len(color_clip) == 2:
                vmin, vmax = map(float, color_clip)

        obs_key = ("metric_tensor" if prune else "energy")
        if invert:
            obs_key += "_density"
        adata.obs[obs_key] = scalar_np

        # ---------- make / reuse UMAP ----------
        if recompute_umap or obsm_key not in adata.obsm:
            sc.pp.neighbors(adata, use_rep="X_pca", random_state=random_state)
            sc.tl.umap(adata, random_state=random_state)
            if obsm_key != "X_umap":
                adata.obsm[obsm_key] = adata.obsm["X_umap"].copy()

        # ---------- KDE‑based density on UMAP (NEW) ----------
        umap_xy = adata.obsm['X_pca']           # shape (N, 2)
        print(umap_xy.shape)
        kde = gaussian_kde(umap_xy.T, bw_method='silverman')
        adata.obs["umap_density"] = kde(umap_xy.T)      # add to .obs

        # ---------- optional subsample for plotting ----------
        if subsample is not None and subsample < adata.n_obs:
            rng = np.random.default_rng(random_state)
            adata_plot = adata[rng.choice(adata.n_obs, subsample, replace=False)]
        else:
            adata_plot = adata

        # ---------- ensure correct key for plotting ----------
        if obsm_key != "X_umap":
            adata_plot.obsm["X_umap"] = adata_plot.obsm[obsm_key]
        plot_kwargs = dict(
            color=[obs_key, "umap_density"],  # three panels
            vmin=[vmin, None],
            vmax=[vmax, None],
            show=show,
            title=[
                f"UMAP coloured by {obs_key}",
                "UMAP point density (KDE)",
            ],
        )
        plot_kwargs.update(scanpy_plot_kwargs)

        sc.pl.umap(adata_plot, **plot_kwargs)

In [24]:
#fix the wandb.run.summary bug?
def remove_all_forward_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()

remove_all_forward_hooks(score_model)
remove_all_forward_hooks(energy_model)
remove_all_forward_hooks(embed_model)
remove_all_forward_hooks(flow_model)

In [25]:
# visualize_energy(
#     embed_model,
#     pre_energy_model,
#     adata,            # AnnData
#     use_rep="X_pca",
#     prune=False,
#     color_clip=("q", 5, 95),  # 1st-99th percentile clipping
#     cmap="inferno",
# )

In [26]:
# visualize_energy(
#     embed_model,
#     energy_model,
#     adata,            # AnnData
#     use_rep="X_pca",
#     prune=False,
#     color_clip=("q", 5, 95),  # 1st-99th percentile clipping
#     cmap="inferno",
# )

In [27]:
# visualize_energy(
#     embed_model,
#     energy_model,
#     adata,            # AnnData
#     use_rep="X_pca",
#     prune=True,
#     color_clip=("q", 5, 95),  # 1st-99th percentile clipping
#     cmap="inferno",
# )

In [28]:
value = holdout.gene
t = holdout.t
num_traj = 10000
# num_traj = 100

print(predict(flow_model, adata_raw, value, conditions, num_traj, t))
print(predict_ctrl(flow_model, adata_raw, value, conditions, num_traj, t)) #DEBUG
print(predict_exact(flow_model, adata_raw, value, conditions, num_traj, t)) #DEBUG

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 5.01 µs
7.728118419647217
8.467123985290527
4.883416175842285


### Results:

t = 18, 24, 36

heldout is (24, tbx16-tbx16l)

PC = 50

---------------

score flow: 7.830566883087158

straight flow: 7.728118419647217

control: 8.483538627624512

exact: 4.868383407592773

In [29]:
#VALIDATE: look at UMAP of distance embeddings

In [30]:
# X = adata.obsm['X_pca']
# X /= adata.uns['std']
# X = torch.from_numpy(X)
# X = X.to(embed_model.get_device())
# adata.obsm['f_X'] = embed_model.embed_net(X).detach().cpu().numpy()

In [31]:
# sc.pp.neighbors(adata, use_rep='f_X')
# sc.tl.umap(adata)
# sc.pl.umap(
#     adata,
#     color="timepoint",
#     # Setting a smaller point size to get prevent overlap
#     size=2,
# )

In [32]:
# sc.pl.umap(
#     adata,
#     color="cell_type",
#     # Setting a smaller point size to get prevent overlap
#     size=2,
# )